# TEST 1: Leveraging LLMs for Feature Generation and Classification

Typically, if our data has $N$ features, we need around $10N$ data items to reach peak performance with classic classifiers like Logistic Regression. Therefore, if our vocabulary has 10,000 words, we would need around 1 million items in the training set to reach peak performance.

An interesting idea regarding this was explored in 2024 in [Balek, V., S'ykora, L., Sklen'ak, V., & Kliegr, T. (2024). LLM-based feature generation from text for interpretable machine learning. ArXiv, abs/2409.07132](https://arxiv.org/abs/2409.07132). The idea is to use an LLM to generate meaningful and interpretable features from text, and then use Logistic Regression for classification.

For example, in the movie plots dataset, we could have features like:
- "Is the protagonist an animal?" (0 or 1)
- "Does the plot indicate psychological suffering?" (0 or 1)

With a reasonable number of these features, our model could make predictions based on meaningful features instead of raw words.

## Objectives
* Perform feature extraction for a particular dataset
* Compare performance and explainability of classifiers with different approaches. 

## Rules

I highlight a few elements of our usual rules:

* You are **NOT ALLOWED** to use AI to generate any code you are asked to make yourself. This includes ChatGPT, CoPilot and all similar generators.
* You are **NOT ALLOWED** to use Google or any other search engine.
* You are **ALLOWED** to use the offical documentations for libraries: 
    * [sklearn](https://scikit-learn.org/)
    * [numpy](https://numpy.org/)
    * [matplotlib](https://matplotlib.org/)
    * [google AI studio](https://aistudio.google.com/)
* You are **ALLOWED** to use previous code from this course as basis.
* You **MUST** use the university's proctoring software to show you are complying with these rules
* This task is **INDIVIDUAL**. DO NOT share your code or results with anyone else.

## Tasks and Deliverables

* At any point, refer to [Balek et al.](https://arxiv.org/abs/2409.07132). 
* Make a well-commented code to solve each one of the tasks below.
* Each task will be evaluated as:
    * Insufficient: task is not done, off-topic, or low-effort
    * In process: task is incomplete, done with a clear conceptual error, or comments 
    * Proficient: everything works and comments are enough to understand what is being done
    * Advanced: everything works, comments are enough to understand what is being done, and code is well organized and formated using functions, dataclasses, and other adequate structures.
* This task should be finished by the end of the class.
* After you are finished, submit the executed notebook in our LMS system.


### 1. Dataset Preparation:
Adapting Balek et al.'s strategy to our movie plot classification case, create a dataset with at least 100 labeled items and at least 5 meaningful features. None of the features can be the class itself ("is this a drama plot?"). Use a clear strategy to avoid exceeding free tier quotas. Store data locally in a format of your choice.

In [1]:
import pandas as pd

df = pd.read_csv('https://raw.githubusercontent.com/tiagoft/NLP/main/wiki_movie_plots_drama_comedy.csv')
df.head(5)

,Plot,Genre
0,The film is about a family who move to the sub...,comedy
1,Before heading out to a baseball game at a nea...,comedy
2,The plot is that of a black woman going to the...,comedy
3,On a beautiful summer day a father and mother ...,drama
4,A thug accosts a girl as she leaves her workpl...,drama


In [2]:
df.shape

(10343, 2)

In [3]:
df_sample = df.sample(n = 150)
df_comedy = df_sample[df_sample['Genre'] == 'comedy']
df_drama = df_sample[df_sample['Genre'] == 'drama']

In [5]:
import os
from dotenv import load_dotenv
import google.generativeai as genai
load_dotenv()
GEMINI_API_KEY = os.getenv('GEMINI_API_KEY')
#GEMINI_API_KEY = # Go to https://aistudio.google.com/ to get a key. DO NOT commit your key to the repository!

# Start the use of the API
genai.configure(api_key=GEMINI_API_KEY)
model = genai.GenerativeModel(model_name="gemini-1.5-flash")

# Make our prompt here
plots_comedy = '\n\nAnother Plot '.join(list(df_comedy['Plot']))
prompt_comedy = f"I have these Plot of comedy movies: {plots_comedy}. Please read them and identify ten plot elements that indicate the movie Genre. Feature example: The protagonist is an animal."

# Use our prompt
response_comedy = model.generate_content(prompt_comedy)

# Print our response
print(response_comedy.text)

Here are ten plot elements from the provided movie plots that strongly indicate the genre is comedy:

1. **Absurdity and heightened reality:** Several plots feature wildly improbable situations (father rebuilding Old Bailey in living room,  talking mule advising a soldier, a cat inheriting a baseball team).  These defy logic and rely on the audience finding humor in the unexpected.

2. **Character-driven humor:** Many plots center on eccentric, flawed, or foolish characters whose personalities and actions drive the comedic situations (W.C. Fields' grumpy character, Stuart Smalley's self-help struggles,  Ron Burgundy's egotism).

3. **Slapstick and physical comedy:**  Several plots hint at physical comedy and slapstick (the melee of news teams,  characters getting repeatedly injured in improbable ways).

4. **Mistaken identities and misunderstandings:**  The confusion arising from mistaken identities is a comedic trope used in multiple plots (the misheard Pope's name,  the con men imper

In [6]:
# Make our prompt here
plots_drama = '\n\nAnother Plot '.join(list(df_drama['Plot']))
prompt_drama = f"I have these Plot of drama movies: {plots_drama}. Please read them and identify ten plot elements that indicate the movie Genre. Feature example: The protagonist is an animal."

# Use our prompt
response_drama = model.generate_content(prompt_drama)

# Print our response
print(response_drama.text)

Based on the provided plot summaries, here are ten plot elements that strongly suggest the genre is **drama**, with some plots leaning into subgenres like crime thriller or biographical drama:


1. **Complex Relationships:** Many plots center on strained, evolving, or complicated relationships (e.g., father-son in *Gandhi My Father*, husband-wife in *Personal Velocity*, romantic entanglements in *The Young and Prodigious T.S. Spivet*).  These relationships drive the narrative and emotional core of the stories.

2. **Internal Conflicts:**  Protagonists often grapple with internal struggles, moral dilemmas, or personal demons (e.g., self-doubt in aspiring actors, guilt over past actions, navigating identity crises).  These internal battles are central to the dramatic tension.

3. **Social Commentary:** Several plots touch on social issues and injustices (e.g., gender inequality in *Gaja Gamini*, class conflict in *The Dressmaker*, poverty and discrimination in *Amna*).  This elevates the

In [10]:
features = [
    'Does the plot feature wildly improbable situations?',
    'Is the protagonist eccentric?',
    'Does the story feature slapsticks?',
    'Does the story feature witty dialogues?',
    'Does the plot feature romantic entanglements?',
    'Does the plot feature moral dilemmas?',
    'Are the characters morally ambiguous?',
    'Does the story focus on inner turmoil?',
    'Does the plot touch on social issues?'
]

for feature in features:
    df_sample[feature] = 0

In [35]:
df_sample = df_sample.reset_index()

In [ ]:
import re
import time

results = {}
features_text = '\n\n - Feature question:'.join(feature for feature in features)

In [39]:

for idx, plot in enumerate(df_sample.tail(5)['Plot']):
    prompt_classification = f"""
    I have the following list of feature questions to describe a movie plot:

    {features}

    Read the plot below and answer every feature question with 0 (No) or 1 (Yes) for this movie.
    Display your answer in a array of 0s and 1s separated by commas.

    {plot}

    """

    response_classification = model.generate_content(prompt_classification)

    # Print our response
    results[idx] = re.findall(r'\b\d+\b', response_classification.text)

    time.sleep(5)

results

ValueError: Invalid operation: The `response.parts` quick accessor requires a single candidate, but but `response.candidates` is empty.
This appears to be caused by a blocked prompt, see `response.prompt_feedback`: block_reason: PROHIBITED_CONTENT


In [40]:
results

{0: ['0', '1', '1', '1', '1', '0', '0', '0', '0'],
 1: ['0', '1', '1', '1', '1', '0', '0', '0', '0'],
 2: ['1', '1', '0', '1', '1', '1', '1', '1', '1'],
 3: ['1', '1', '0', '1', '1', '1', '1', '1', '1'],
 4: ['0', '1', '0', '0', '0', '1', '1', '1', '0'],
 5: ['0', '0', '0', '0', '1', '1', '1', '1', '0'],
 6: ['0', '0', '0', '0', '1', '0', '0', '0', '0'],
 7: ['0', '0', '0', '1', '1', '1', '1', '1', '0'],
 8: ['0', '1', '1', '1', '0', '0', '0', '0', '0'],
 9: ['0', '0', '0', '0', '0', '1', '0', '0', '0'],
 10: ['0', '0', '0', '0', '0', '1', '1', '1', '0'],
 11: ['0', '0', '0', '0', '1', '0', '0', '1', '0'],
 12: ['0', '0', '0', '0', '1', '0', '1', '1', '0'],
 13: ['0', '0', '0', '0', '0', '0', '0', '1', '0'],
 14: ['0', '0', '0', '0', '1', '1', '1', '1', '1'],
 15: ['0', '0', '1', '0', '1', '0', '0', '0', '0'],
 16: ['0', '0', '0', '0', '0', '0', '0', '1', '1'],
 17: ['1', '1', '1', '1', '1', '1', '1', '1', '0'],
 18: ['0', '0', '0', '0', '0', '0', '0', '0', '0'],
 19: ['1', '1', '1', '

In [45]:
df_results = pd.DataFrame(results).transpose()
df_results.columns = features
df_results.shape

(147, 9)

In [52]:
df_results.to_csv("classification.csv")

In [ ]:
df_sample = df_sample.loc[:146, ]

In [58]:
df_sample = df_sample.loc[:, ['Plot', 'Genre']]
df_sample.to_csv('sample.csv')

### 2. Classification:
Use the generated features to train a Logistic Regression model. Use cross-validation to select the best hyperparameters. Report accuracy and f1-score for your classifier.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.pipeline import Pipeline

model_lr = Pipeline([
    ('classifier', LogisticRegression())
])

In [54]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

X_train, X_test, y_train, y_test = train_test_split(df_results, df_sample['Genre'], test_size=0.2)
model_lr.fit(X_train, y_train)

Pipeline(steps=[('classifier', LogisticRegression())])

In [55]:
y_pred = model_lr.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f'Accuracy: {accuracy:.2f}')

Accuracy: 0.80


### 3. Performance Comparison
Compare the performance of the following approaches:
1. Traditional Bag-of-Words
2. LLM-generated features with Logistic Regression
3. Direct classification using LLM

Use a bar plot to show the performance differences (choose either accuracy or F1-score).

In [64]:
from sklearn.naive_bayes import BernoulliNB
from sklearn.feature_extraction.text import TfidfVectorizer

model_bof = Pipeline([
    ('vectorizer', CountVectorizer(binary=True, stop_words='english')),
    ('classifier', BernoulliNB())
])

X_train_bof, X_test_bof, y_train_bof, y_test_bof = train_test_split(df_sample['Plot'], df_sample['Genre'], test_size=0.2)
model_bof.fit(X_train_bof, y_train_bof)

Pipeline(steps=[('vectorizer',
                 CountVectorizer(binary=True, stop_words='english')),
                ('classifier', BernoulliNB())])

In [65]:
y_pred_bof = model_bof.predict(X_test_bof)
accuracy_bof = accuracy_score(y_test_bof, y_pred_bof)
print(f'Accuracy: {accuracy_bof:.2f}')

Accuracy: 0.67


### 4. Improvement Strategies
Determine whether labeling more items would improve system performance. Use data to justify your answer.